# NYC Taxi Trip Dataset — Complete EDA Practical (Units 1–5)

This Colab notebook uses **one NYC taxi CSV** to cover the tasks in all five syllabus units:
1. Exploratory Data Analysis and data preparation
2. Matplotlib, Seaborn, Plotly, and Bokeh visualizations
3. Univariate analysis
4. Bivariate analysis
5. Multivariate analysis, PCA, and a Scikit-learn model

**How to use:** Run cells from top to bottom. When prompted, upload your CSV file. If the CSV is huge, the notebook includes an optional sampling setting for plotting/modeling.

The screenshots show columns such as `tpep_pickup_datetime`, `tpep_dropoff_datetime`, `passenger_count`, `trip_distance`, `fare_amount`, `tip_amount`, and `total_amount`. The code normalizes column names and checks for optional columns, so it can handle small naming differences.

## 0. Install and import libraries

In [ ]:
!pip -q install seaborn plotly bokeh scikit-learn

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)


## 1. Load the CSV in Google Colab

In [ ]:
from google.colab import files
uploaded = files.upload()  # Choose your taxi CSV file

csv_name = next(iter(uploaded))
# For very large CSVs, consider adding nrows=500_000 to read_csv.
df_raw = pd.read_csv(csv_name, low_memory=False)

print("File:", csv_name)
print("Shape:", df_raw.shape)
display(df_raw.head())
print("\\nColumn names:")
print(df_raw.columns.tolist())


## UNIT 1 — Exploratory Data Analysis

### 1.1 Data Science process and first inspection

In [ ]:
print("Rows, columns:", df_raw.shape)
display(df_raw.head(10))
display(df_raw.sample(min(5, len(df_raw)), random_state=42))
df_raw.info()
display(df_raw.describe(include="all").T)


### 1.2 Normalize column names and parse dates

In [ ]:
df = df_raw.copy()

# Normalize column names (e.g. tpep_pickup_datetime remains easy to reference)
df.columns = (df.columns.str.strip().str.lower()
              .str.replace(r"[^a-z0-9]+", "_", regex=True)
              .str.strip("_"))

# Parse taxi timestamps if present
for col in ["tpep_pickup_datetime", "tpep_dropoff_datetime",
            "lpep_pickup_datetime", "lpep_dropoff_datetime",
            "pickup_datetime", "dropoff_datetime"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

# Derive helpful analysis features when timestamps exist
pickup_col = next((c for c in ["tpep_pickup_datetime", "lpep_pickup_datetime", "pickup_datetime"] if c in df.columns), None)
dropoff_col = next((c for c in ["tpep_dropoff_datetime", "lpep_dropoff_datetime", "dropoff_datetime"] if c in df.columns), None)

if pickup_col:
    df["pickup_hour"] = df[pickup_col].dt.hour
    df["pickup_dayofweek"] = df[pickup_col].dt.day_name()
    df["pickup_month"] = df[pickup_col].dt.month
if pickup_col and dropoff_col:
    df["trip_duration_min"] = (df[dropoff_col] - df[pickup_col]).dt.total_seconds() / 60

display(df.head())


### 1.3 Missing values, duplicates, and data quality

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
missing_table = pd.DataFrame({"missing_count": missing, "missing_percent": missing_pct})
display(missing_table[missing_table["missing_count"] > 0])

print("Duplicate rows:", df.duplicated().sum())

# Visualize missingness
missing_top = missing[missing > 0].head(20)
if len(missing_top):
    missing_top.sort_values().plot(kind="barh", figsize=(9, 5), title="Missing values by column (top 20)")
    plt.xlabel("Number of missing values")
    plt.show()
else:
    print("No missing values detected.")


### 1.4 Clean data, handle outliers, and encode categories

In [ ]:
# Remove exact duplicate rows
df_clean = df.drop_duplicates().copy()

# Convert numeric-looking columns to numeric when appropriate
for col in df_clean.columns:
    if col not in [pickup_col, dropoff_col, "pickup_dayofweek"]:
        # Keep object columns categorical unless values are mostly numeric
        if df_clean[col].dtype == "object":
            converted = pd.to_numeric(df_clean[col], errors="coerce")
            if converted.notna().mean() > 0.9:
                df_clean[col] = converted

# Remove impossible/invalid values for common taxi fields, if present
validity_rules = {
    "passenger_count": lambda s: s.between(0, 9),
    "trip_distance": lambda s: s.ge(0),
    "fare_amount": lambda s: s.ge(0),
    "total_amount": lambda s: s.ge(0),
    "trip_duration_min": lambda s: s.between(0, 24*60)
}
for col, rule in validity_rules.items():
    if col in df_clean.columns:
        df_clean = df_clean[df_clean[col].isna() | rule(df_clean[col])]

print("Before cleaning:", df.shape)
print("After duplicate/validity cleaning:", df_clean.shape)

# Median imputation for numeric columns and mode for categorical columns (EDA copy)
df_eda = df_clean.copy()
num_cols_all = df_eda.select_dtypes(include=np.number).columns.tolist()
cat_cols_all = df_eda.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

for col in num_cols_all:
    df_eda[col] = df_eda[col].fillna(df_eda[col].median())
for col in cat_cols_all:
    mode_vals = df_eda[col].mode(dropna=True)
    if len(mode_vals):
        df_eda[col] = df_eda[col].fillna(mode_vals.iloc[0])

# One-hot encoding example: do not replace the main data frame
categorical_examples = [c for c in ["store_and_fwd_flag", "payment_type"] if c in df_eda.columns]
if categorical_examples:
    encoded_example = pd.get_dummies(df_eda[categorical_examples].astype("category"), prefix=categorical_examples, dtype=int)
    display(encoded_example.head())

display(df_eda.head())


### 1.5 Slicing, transformation, and grouping

In [ ]:
# Data set slicing examples
display(df_eda.iloc[:10, :8])  # first 10 rows, first 8 columns
if "payment_type" in df_eda.columns:
    display(df_eda[df_eda["payment_type"].astype(str) == df_eda["payment_type"].astype(str).iloc[0]].head())

# Transformations: derived variables, log transformation for skewed positive values
if "trip_distance" in df_eda.columns:
    df_eda["log_trip_distance"] = np.log1p(pd.to_numeric(df_eda["trip_distance"], errors="coerce").clip(lower=0))
if "total_amount" in df_eda.columns:
    df_eda["cost_per_mile"] = (
        pd.to_numeric(df_eda["total_amount"], errors="coerce") /
        pd.to_numeric(df_eda.get("trip_distance", np.nan), errors="coerce").replace(0, np.nan)
    )

# Grouping examples
group_col = "payment_type" if "payment_type" in df_eda.columns else None
measure_col = "total_amount" if "total_amount" in df_eda.columns else ("fare_amount" if "fare_amount" in df_eda.columns else None)
if group_col and measure_col:
    grouped = df_eda.groupby(group_col, dropna=False)[measure_col].agg(["count", "mean", "median", "std"]).sort_values("count", ascending=False)
    display(grouped)


## UNIT 2 — Visualizing using Matplotlib, Seaborn, Plotly, and Bokeh
For performance, visualizations use a sample of at most 100,000 rows. Change `PLOT_SAMPLE_SIZE` if your computer can handle more.

In [ ]:
PLOT_SAMPLE_SIZE = 100_000
plot_df = df_eda.sample(min(PLOT_SAMPLE_SIZE, len(df_eda)), random_state=42).copy()

def has_cols(*cols):
    return all(c in plot_df.columns for c in cols)

# 2.1 Simple line plot: average trip distance by pickup hour
if has_cols("pickup_hour", "trip_distance"):
    line_data = plot_df.groupby("pickup_hour", dropna=True)["trip_distance"].mean()
    plt.figure(figsize=(9, 4))
    plt.plot(line_data.index, line_data.values, marker="o")
    plt.title("Average Trip Distance by Pickup Hour")
    plt.xlabel("Pickup hour"); plt.ylabel("Average trip distance")
    plt.grid(True); plt.show()

# 2.2 Scatter plot
if has_cols("trip_distance", "fare_amount"):
    plt.figure(figsize=(7, 5))
    plt.scatter(plot_df["trip_distance"], plot_df["fare_amount"], alpha=.25)
    plt.title("Trip Distance vs Fare Amount")
    plt.xlabel("Trip distance"); plt.ylabel("Fare amount")
    plt.show()

# 2.3 Visualizing errors: mean fare by payment type with standard-deviation error bars
if has_cols("payment_type", "fare_amount"):
    err = plot_df.groupby("payment_type")["fare_amount"].agg(["mean", "std", "count"]).dropna()
    err = err[err["count"] >= 2].head(10)
    plt.figure(figsize=(8, 4))
    plt.errorbar(err.index.astype(str), err["mean"], yerr=err["std"], fmt="o", capsize=4)
    plt.title("Mean Fare by Payment Type (±1 Standard Deviation)")
    plt.xlabel("Payment type"); plt.ylabel("Fare amount")
    plt.xticks(rotation=30); plt.show()

# 2.4 Density plot and contour plot
if has_cols("trip_distance", "fare_amount"):
    sns.kdeplot(data=plot_df, x="trip_distance", fill=True)
    plt.title("Density of Trip Distance"); plt.show()

    contour_df = plot_df[["trip_distance", "fare_amount"]].replace([np.inf, -np.inf], np.nan).dropna()
    contour_df = contour_df[(contour_df["trip_distance"] >= 0) & (contour_df["fare_amount"] >= 0)]
    contour_df = contour_df.sample(min(15000, len(contour_df)), random_state=42)
    if len(contour_df) > 100:
        plt.figure(figsize=(7, 5))
        sns.kdeplot(data=contour_df, x="trip_distance", y="fare_amount", levels=8, cmap="viridis")
        plt.title("Contour Plot: Trip Distance vs Fare")
        plt.show()

# 2.5 Histogram, legends, colors, subplots, annotations
numeric_for_hist = [c for c in ["trip_distance", "fare_amount", "tip_amount", "total_amount"] if c in plot_df.columns]
if numeric_for_hist:
    plot_df[numeric_for_hist].hist(figsize=(12, 7), bins=30, color=None)
    plt.suptitle("Histograms of Taxi Variables")
    plt.tight_layout(); plt.show()

if has_cols("trip_distance", "total_amount"):
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(plot_df["trip_distance"], plot_df["total_amount"], alpha=.25, label="Taxi trips")
    ax.set_title("Trip Distance vs Total Amount")
    ax.set_xlabel("Trip distance"); ax.set_ylabel("Total amount")
    ax.legend()
    # Annotation on median point
    med_x, med_y = plot_df["trip_distance"].median(), plot_df["total_amount"].median()
    ax.annotate("Median", (med_x, med_y), xytext=(med_x + 1, med_y + 10),
                arrowprops={"arrowstyle": "->"})
    plt.show()

# 2.6 3D plotting
if has_cols("trip_distance", "fare_amount", "tip_amount"):
    d3 = plot_df[["trip_distance", "fare_amount", "tip_amount"]].dropna()
    d3 = d3.sample(min(3000, len(d3)), random_state=42)
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection="3d")
    ax.scatter(d3["trip_distance"], d3["fare_amount"], d3["tip_amount"], alpha=.35)
    ax.set_xlabel("Trip distance"); ax.set_ylabel("Fare"); ax.set_zlabel("Tip")
    ax.set_title("3D Taxi Trip Plot"); plt.show()


### 2.7 Seaborn, Plotly, and Bokeh

In [ ]:
# Seaborn grouped plot
if has_cols("payment_type", "total_amount"):
    sns.boxplot(data=plot_df, x="payment_type", y="total_amount")
    plt.title("Total Amount by Payment Type")
    plt.xlabel("Payment type"); plt.ylabel("Total amount")
    plt.show()

# Plotly interactive scatter
try:
    import plotly.express as px
    if has_cols("trip_distance", "fare_amount"):
        p = plot_df[["trip_distance", "fare_amount"]].dropna().sample(
            min(5000, len(plot_df[["trip_distance", "fare_amount"]].dropna())), random_state=42
        )
        fig = px.scatter(p, x="trip_distance", y="fare_amount",
                         title="Interactive: Trip Distance vs Fare")
        fig.show()
except Exception as e:
    print("Plotly plot skipped:", e)

# Bokeh interactive scatter
try:
    from bokeh.plotting import figure, show, output_notebook
    from bokeh.models import ColumnDataSource
    output_notebook()
    if has_cols("trip_distance", "fare_amount"):
        b = plot_df[["trip_distance", "fare_amount"]].dropna().sample(
            min(3000, len(plot_df[["trip_distance", "fare_amount"]].dropna())), random_state=42
        )
        p = figure(title="Bokeh: Trip Distance vs Fare", x_axis_label="Trip distance",
                   y_axis_label="Fare amount", width=700, height=400)
        p.circle(b["trip_distance"], b["fare_amount"], size=5, alpha=.35)
        show(p)
except Exception as e:
    print("Bokeh plot skipped:", e)


## UNIT 3 — Univariate Analysis

### 3.1 Summary statistics: mean, median, standard deviation, quartiles

In [ ]:
numeric_cols = df_eda.select_dtypes(include=np.number).columns.tolist()
display(df_eda[numeric_cols].describe().T)

# Explicit statistics for key columns
key_stats = [c for c in ["trip_distance", "fare_amount", "tip_amount", "total_amount", "passenger_count", "trip_duration_min"] if c in df_eda.columns]
if key_stats:
    stats = pd.DataFrame({
        "mean": df_eda[key_stats].mean(),
        "median": df_eda[key_stats].median(),
        "std_dev": df_eda[key_stats].std(),
        "Q1": df_eda[key_stats].quantile(.25),
        "Q3": df_eda[key_stats].quantile(.75),
        "IQR": df_eda[key_stats].quantile(.75) - df_eda[key_stats].quantile(.25),
        "skewness": df_eda[key_stats].skew()
    })
    display(stats)


### 3.2 Histograms, box plots, density plots, skewness, and outliers

In [ ]:
for col in [c for c in ["trip_distance", "fare_amount", "tip_amount", "total_amount", "trip_duration_min"] if c in df_eda.columns]:
    s = pd.to_numeric(df_eda[col], errors="coerce").dropna()
    if len(s) == 0:
        continue
    fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
    sns.histplot(s.sample(min(100000, len(s)), random_state=42), kde=False, ax=axes[0])
    axes[0].set_title(f"Histogram: {col}")
    sns.boxplot(x=s.sample(min(100000, len(s)), random_state=42), ax=axes[1])
    axes[1].set_title(f"Box plot: {col}")
    sns.kdeplot(x=s.sample(min(50000, len(s)), random_state=42), fill=True, ax=axes[2])
    axes[2].set_title(f"Density: {col}")
    plt.tight_layout(); plt.show()
    print(col, "| skewness =", round(s.skew(), 3))
    q1, q3 = s.quantile(.25), s.quantile(.75)
    iqr = q3 - q1
    outlier_count = ((s < q1 - 1.5*iqr) | (s > q3 + 1.5*iqr)).sum()
    print("IQR outlier count:", int(outlier_count))


### 3.3 Categorical variables: frequency tables, bar charts, and pie charts

In [ ]:
categorical_candidates = [c for c in ["payment_type", "passenger_count", "store_and_fwd_flag", "RatecodeID", "ratecodeid"] if c in df_eda.columns]
for col in categorical_candidates[:4]:
    counts = df_eda[col].astype(str).value_counts(dropna=False).head(15)
    display(counts.rename_axis(col).reset_index(name="frequency"))
    counts.plot(kind="bar", figsize=(8, 3), title=f"Frequency by {col}")
    plt.xlabel(col); plt.ylabel("Frequency"); plt.xticks(rotation=30); plt.show()

# Pie chart for payment type if available
if "payment_type" in df_eda.columns:
    counts = df_eda["payment_type"].astype(str).value_counts().head(8)
    counts.plot(kind="pie", autopct="%1.1f%%", figsize=(6, 6), ylabel="", title="Payment Type Share")
    plt.show()


## UNIT 4 — Bivariate Analysis

### 4.1 Relationships between two numerical variables: scatter, Pearson, and Spearman

In [ ]:
biv_candidates = [c for c in ["trip_distance", "fare_amount", "tip_amount", "total_amount", "passenger_count", "trip_duration_min"] if c in df_eda.columns]
if len(biv_candidates) >= 2:
    pearson = df_eda[biv_candidates].corr(method="pearson", numeric_only=True)
    spearman = df_eda[biv_candidates].corr(method="spearman", numeric_only=True)
    print("Pearson correlation:")
    display(pearson)
    print("Spearman correlation:")
    display(spearman)

if "trip_distance" in df_eda.columns and "fare_amount" in df_eda.columns:
    s = df_eda[["trip_distance", "fare_amount"]].dropna()
    s = s.sample(min(50000, len(s)), random_state=42)
    sns.regplot(data=s, x="trip_distance", y="fare_amount", scatter_kws={"alpha": .2, "s": 10}, line_kws={"color": "red"})
    plt.title("Trip Distance vs Fare with Trend Line"); plt.show()


### 4.2 Two categorical variables: grouped and stacked bar charts

In [ ]:
# Example: payment type versus passenger count (passenger count treated as category)
if "payment_type" in df_eda.columns and "passenger_count" in df_eda.columns:
    temp = df_eda[["payment_type", "passenger_count"]].dropna().copy()
    temp["passenger_count"] = temp["passenger_count"].astype(str)
    ct = pd.crosstab(temp["payment_type"].astype(str), temp["passenger_count"])
    display(ct)
    ct.plot(kind="bar", figsize=(10, 4), title="Grouped Bar: Payment Type by Passenger Count")
    plt.xlabel("Payment type"); plt.ylabel("Trip count"); plt.xticks(rotation=0); plt.show()
    ct.plot(kind="bar", stacked=True, figsize=(10, 4), title="Stacked Bar: Payment Type by Passenger Count")
    plt.xlabel("Payment type"); plt.ylabel("Trip count"); plt.xticks(rotation=0); plt.show()
else:
    print("Need payment_type and passenger_count columns for this example.")


### 4.3 Numerical versus categorical: box plots and violin plots

In [ ]:
category_col = "payment_type" if "payment_type" in df_eda.columns else None
numeric_col = "total_amount" if "total_amount" in df_eda.columns else ("fare_amount" if "fare_amount" in df_eda.columns else None)
if category_col and numeric_col:
    temp = df_eda[[category_col, numeric_col]].dropna().copy()
    temp[category_col] = temp[category_col].astype(str)
    # Limit category count and extreme y-axis values for readable visualization
    keep = temp[category_col].value_counts().head(8).index
    temp = temp[temp[category_col].isin(keep)]
    upper = temp[numeric_col].quantile(.99)
    temp = temp[temp[numeric_col] <= upper]
    plt.figure(figsize=(9, 4))
    sns.boxplot(data=temp, x=category_col, y=numeric_col)
    plt.title(f"Box Plot: {numeric_col} by {category_col} (up to 99th percentile)")
    plt.show()
    plt.figure(figsize=(9, 4))
    sns.violinplot(data=temp.sample(min(20000, len(temp)), random_state=42), x=category_col, y=numeric_col, cut=0)
    plt.title(f"Violin Plot: {numeric_col} by {category_col}")
    plt.show()


## UNIT 5 — Multivariate Analysis and Insights

### 5.1 Pair plot (scatter matrix) and correlation heatmap

In [ ]:
pair_cols = [c for c in ["trip_distance", "fare_amount", "tip_amount", "total_amount", "passenger_count"] if c in df_eda.columns]
if len(pair_cols) >= 2:
    pair_sample = df_eda[pair_cols].dropna().sample(min(1500, len(df_eda[pair_cols].dropna())), random_state=42)
    sns.pairplot(pair_sample, corner=True, diag_kind="hist")
    plt.show()

corr_cols = df_eda.select_dtypes(include=np.number).columns.tolist()
corr_cols = [c for c in corr_cols if df_eda[c].nunique(dropna=True) > 1][:25]
if len(corr_cols) >= 2:
    plt.figure(figsize=(12, 9))
    sns.heatmap(df_eda[corr_cols].corr(), cmap="coolwarm", center=0, annot=False)
    plt.title("Correlation Heatmap")
    plt.show()


### 5.2 Dimensionality reduction with PCA

In [ ]:
# PCA is performed on numeric features only; median-impute and standardize first.
pca_cols = [c for c in ["passenger_count", "trip_distance", "fare_amount", "extra", "mta_tax",
                        "tip_amount", "tolls_amount", "improvement_surcharge", "total_amount",
                        "congestion_surcharge", "trip_duration_min", "pickup_hour"] if c in df_eda.columns]
pca_data = df_eda[pca_cols].replace([np.inf, -np.inf], np.nan).copy() if pca_cols else pd.DataFrame()

if len(pca_cols) >= 2:
    pca_data = pca_data.apply(pd.to_numeric, errors="coerce")
    pca_data = pca_data.fillna(pca_data.median()).fillna(0)
    pca_sample = pca_data.sample(min(50000, len(pca_data)), random_state=42)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(pca_sample)
    pca = PCA(n_components=2, random_state=42)
    X_pca = pca.fit_transform(X_scaled)
    print("Explained variance ratio:", pca.explained_variance_ratio_)
    print("Total variance explained by first 2 components:", round(pca.explained_variance_ratio_.sum(), 4))
    plt.figure(figsize=(7, 5))
    plt.scatter(X_pca[:, 0], X_pca[:, 1], alpha=.25, s=10)
    plt.xlabel("Principal Component 1"); plt.ylabel("Principal Component 2")
    plt.title("PCA Projection of Taxi Trips"); plt.show()
else:
    print("Not enough numeric columns available for PCA.")


### 5.3 Scikit-learn model development: predict long trips
We define a long trip as trip duration above the dataset median. This avoids needing a competition target label. Features that directly reveal duration are excluded from predictors to prevent target leakage.

In [ ]:
# Prefer duration derived from pickup/dropoff timestamps; otherwise use a regression target such as total_amount.
if "trip_duration_min" in df_eda.columns:
    model_df = df_eda.copy()
    model_df = model_df[model_df["trip_duration_min"].between(1, 240)].copy()
    model_df["long_trip"] = (model_df["trip_duration_min"] > model_df["trip_duration_min"].median()).astype(int)

    # Exclude raw timestamps, duration, and derived target from features
    leakage_cols = {"trip_duration_min", "long_trip"}
    if pickup_col: leakage_cols.add(pickup_col)
    if dropoff_col: leakage_cols.add(dropoff_col)
    # Exclude identifiers that are not useful predictive features
    leakage_cols.update([c for c in ["vendorid", "pulocationid", "dolocationid"] if c in model_df.columns])

    feature_cols = [c for c in model_df.columns if c not in leakage_cols]
    # Drop columns with too many missing values or a single value
    feature_cols = [c for c in feature_cols if model_df[c].notna().mean() >= 0.5 and model_df[c].nunique(dropna=True) > 1]
    X = model_df[feature_cols].copy()
    y = model_df["long_trip"]

    # Convert datetimes not excluded above to useful categorical/numeric features
    for c in X.columns:
        if pd.api.types.is_datetime64_any_dtype(X[c]):
            X[c] = X[c].astype(str)

    numeric_features = X.select_dtypes(include=np.number).columns.tolist()
    categorical_features = [c for c in X.columns if c not in numeric_features]

    numeric_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")),
                             ("scaler", StandardScaler())])
    categorical_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                                 ("onehot", OneHotEncoder(handle_unknown="ignore"))])
    preprocessor = ColumnTransformer([
        ("num", numeric_pipe, numeric_features),
        ("cat", categorical_pipe, categorical_features)
    ])

    # Sampling keeps training practical on Colab; increase if desired.
    max_model_rows = 150_000
    if len(X) > max_model_rows:
        chosen = model_df.sample(max_model_rows, random_state=42).index
        X = X.loc[chosen]
        y = y.loc[chosen]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    model = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1,
                                              class_weight="balanced_subsample"))
    ])
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    print("Accuracy:", round(accuracy_score(y_test, pred), 4))
    print(classification_report(y_test, pred, zero_division=0))
    print("Confusion matrix:\\n", confusion_matrix(y_test, pred))
else:
    print("Pickup/dropoff timestamp columns are required to derive trip duration for this classification task.")
    print("Alternative: use fare_amount or total_amount as a regression target.")


## Final report checklist

Use these points in your record/report:
- **Unit 1:** State the original shape, data types, missing values, duplicate count, cleaning rules, encoding, slicing, transformations, and group-by result.
- **Unit 2:** Insert screenshots of line, scatter, error-bar, density/contour, histogram, annotated/subplot, 3D, Seaborn, Plotly, and Bokeh outputs.
- **Unit 3:** Report mean, median, standard deviation, quartiles, IQR, skewness, outlier counts, and categorical frequencies.
- **Unit 4:** Compare Pearson vs Spearman correlations, categorical frequency charts, and numeric distributions by category.
- **Unit 5:** Explain patterns in the pair plot/heatmap, report PCA explained variance, and evaluate the model using accuracy and the classification report.
- **Limitations:** Taxi data can contain invalid trips, outliers, and unusual fare values. State your cleaning choices; do not claim correlations prove causation.